<a href="https://colab.research.google.com/github/Sanjaysanjay31/24B11DS095_DriveReady_OPC_2026/blob/main/Day_5/Startup_Marketing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 — Memory and RAG: An Agent That Remembers and Looks Things Up

**Project:** AI Startup Marketing Agent  
**Primary Provider:** OpenRouter — NVIDIA Nemotron 3.5 Super VL (Free)  
**Fallback Provider:** Groq — GPT-OSS 120B  
**Database:** [Google Sheets Campaign Orders](https://docs.google.com/spreadsheets/d/1WryffEEH-N9TpWcwg8pvdaNU0XGzQWqBRS2Lnt0Ro90)  

On Day 4, our agent gained autonomous tools and Google Sheets logging. Today, we achieve full single-agent sign-off with:
1. **Conversational Memory with Checkpointing:** Maintaining distinct founder conversations across sessions using LangGraph `InMemorySaver` and `thread_id`.
2. **Retrieval-Augmented Generation (RAG):** Indexing agency policy documents and knowledge into a vector database (ChromaDB) to ground agent responses in factual specifications.
3. **Grounded Tool Calling & Guardrails:** Combining pricing, operating hours, sheet logging, and vector search with strict out-of-scope guardrails.

In [1]:
!pip install -q langchain langchain-groq langchain-openrouter chromadb gspread google-auth python-dotenv

import os

# Load environment variables from .env locally or Colab Secrets
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    from google.colab import userdata
    colab_openrouter = userdata.get("OPENROUTER_API_KEY")
    colab_groq = userdata.get("GROQ_API_KEY") or userdata.get("agentic_ai_GROQ_API_KEY")
except Exception:
    colab_openrouter = None
    colab_groq = None

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or colab_openrouter
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or colab_groq

from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter

# Primary LLM: OpenRouter (NVIDIA Nemotron 3.5 Super VL) | Fallback: Groq (GPT-OSS 120B)
openrouter_llm = ChatOpenRouter(model="nvidia/nemotron-3.5-super-vl:free", temperature=0.1, api_key=OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None
groq_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1, max_retries=3, api_key=GROQ_API_KEY) if GROQ_API_KEY else None

if openrouter_llm and groq_llm:
    llm = openrouter_llm.with_fallbacks([groq_llm])
else:
    llm = openrouter_llm or groq_llm

# Google Sheets connection for confirmed campaign bookings
ws = None
SPREADSHEET_ID = "1WryffEEH-N9TpWcwg8pvdaNU0XGzQWqBRS2Lnt0Ro90"

try:
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SPREADSHEET_ID)
    ws = sh.sheet1
    print("Connected to Google Sheet in Colab:", sh.title, f"({sh.url})")
except Exception:
    try:
        import gspread
        gc = gspread.service_account(filename="google_service_account.json")
        sh = gc.open_by_key(SPREADSHEET_ID)
        ws = sh.sheet1
        print("Connected to Google Sheet via Service Account:", sh.title)
    except Exception:
        print("Notice: Running locally without Google credentials. Orders will be validated and logged locally.")

print("Setup complete! Model and Google Sheets integration ready.")

Connected to Google Sheet: Sweet Crumbs Orders (https://docs.google.com/spreadsheets/d/1WryffEEH-N9TpWcwg8pvdaNU0XGzQWqBRS2Lnt0Ro90)
Setup complete! Model and Google Sheets integration ready.


In [2]:
# Knowledge Base for AI Startup Marketing Agency (14 Core Policy & Technical Facts)
FAQS = {
    "revisions": "Every video order includes 2 rounds of free script and visual revisions. Additional revisions are Rs 150 each.",
    "turnaround": "Standard turnaround time is 48 hours for script and audio, and 72 hours for full vertical video assembly.",
    "aspect_ratio": "All videos are produced in 9:16 vertical format (1080x1920 resolution at 60 fps), optimized for YouTube Shorts, Instagram Reels, and TikTok.",
    "audio_licensing": "All background music tracks and sound effects used are 100% royalty-free and cleared for commercial monetization on YouTube and Meta.",
    "nda_confidentiality": "We sign a mutual Non-Disclosure Agreement (NDA) on request. Your unreleased product features, pitch decks, and roadmaps remain strictly confidential.",
    "cancellation": "Cancellations are 100% free before script drafting begins. Once video rendering starts, a 50% kill fee applies to cover GPU compute costs.",
    "voice_cloning": "Custom founder voice cloning requires 3 minutes of clean audio sample and takes 24 hours to calibrate.",
    "thumbnails": "Every vertical video package includes 2 high-CTR vertical thumbnail frames (cover images) at no extra charge.",
    "script_approval": "We do not render final videos until the founder reviews and approves the hook and narration script.",
    "retention_guarantee": "Our scripts follow the 3-second hook rule designed to achieve over 70% retention in the first 5 seconds.",
    "bulk_discount": "Orders of 10 or more vertical videos receive a 15% discount upon agency founder approval.",
    "subtitles": "All videos include stylized dynamic karaoke-style subtitles synced to speech with custom brand colors.",
    "deliverables": "Final deliverables include master .mp4 video files, SRT caption tracks, and individual clean audio voiceover stems.",
    "payment_terms": "We accept UPI, debit/credit cards, and bank transfer. Custom retainer packages require a 50% advance."
}

# Text chunking function with sliding window overlap (as demonstrated in RAG fundamentals)
def split_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

# Indexing into ChromaDB vector database (with graceful fallback for local execution)
try:
    import chromadb
    client = chromadb.Client()
    faq_collection = client.get_or_create_collection(name="startup_marketing_faq")
    faq_collection.upsert(ids=list(FAQS.keys()), documents=list(FAQS.values()))
    print(f"ChromaDB initialized: {faq_collection.count()} policy facts indexed in vector database.")
except Exception:
    class FallbackVectorStore:
        def __init__(self, data):
            self.data = data
        def query(self, query_texts, n_results=3):
            q_words = set(query_texts[0].lower().split())
            scored = []
            for doc in self.data.values():
                score = len(q_words & set(doc.lower().split()))
                scored.append((score, doc))
            scored.sort(key=lambda x: x[0], reverse=True)
            return {"documents": [[doc for _, doc in scored[:n_results]]]}
    faq_collection = FallbackVectorStore(FAQS)
    print("Fallback vector store ready for local execution.")

# Verify chunking and vector retrieval
policy_sample = " ".join(list(FAQS.values())[:3])
print(f"Chunking test: {len(split_text(policy_sample))} chunks created from full policy text.")

test_query = "What video aspect ratio and resolution do you deliver?"
retrieved = faq_collection.query(query_texts=[test_query], n_results=2)
print(f"Sample query: '{test_query}'")
print("Retrieved fact:", retrieved["documents"][0][0])

ChromaDB initialized: 14 policy facts indexed in vector database.
Chunking test: 3 chunks created from full policy text.
Sample query: 'What video aspect ratio and resolution do you deliver?'
Retrieved fact: All videos are produced in 9:16 vertical format (1080x1920 resolution at 60 fps), optimized for YouTube Shorts, Instagram Reels, and TikTok.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Service catalogue and pipeline operating schedule
SERVICES = {
    "viral hook generation": 400,
    "short script writing": 700,
    "storyboard generation": 1200,
    "voiceover synthesis": 500,
    "auto subtitle karaoke": 300,
    "vertical video assembly": 1500,
    "seo metadata pack": 350
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "10 am to 6 pm",
    "sunday": "10 am to 2 pm"
}

class CampaignOrder(BaseModel):
    """A short-form video marketing campaign order placed by a founder."""
    client_name: str = Field(description="Founder's full or first name")
    startup_name: str = Field(description="Name of the startup product or company")
    service: Literal[
        "viral hook generation",
        "short script writing",
        "storyboard generation",
        "voiceover synthesis",
        "auto subtitle karaoke",
        "vertical video assembly",
        "seo metadata pack"
    ] = Field(description="Selected marketing service from the catalogue")
    platform: Literal["YouTube Shorts", "Instagram Reels", "TikTok", "All"] = Field(description="Target video platform")
    target_date: str = Field(description="Requested delivery or launch date as stated by the founder")

@tool
def get_service_price(service: str, qty: int = 1) -> str:
    """Get the total price in rupees for a marketing service from the catalogue.
    Args:
        service: Name of the service, e.g. 'viral hook generation', 'short script writing', or 'vertical video assembly'
        qty: Number of videos or service units requested
    """
    price = SERVICES.get(service.lower().strip())
    if price is None:
        available = ", ".join(SERVICES.keys())
        return f"'{service}' is not in our catalogue. Available services: {available}."
    total = price * qty
    return f"{qty} x {service} = Rs {total}"

@tool
def check_operating_hours(day: str) -> str:
    """Check the agency and rendering pipeline operating hours for a day of the week, e.g. 'sunday'."""
    hours = HOURS.get(day.lower().strip())
    return f"Support & video pipeline open {hours} on {day}." if hours else f"'{day}' is not a valid day of the week."

@tool
def log_campaign_order(client_name: str, startup_name: str, service: str, platform: str, target_date: str) -> str:
    """Save a confirmed campaign order into the Google Sheets database.
    Use ONLY after confirming the client's name, startup name, service, platform, and delivery date."""
    try:
        order = CampaignOrder(
            client_name=client_name.strip(),
            startup_name=startup_name.strip(),
            service=service.lower().strip(),
            platform=platform,
            target_date=target_date.strip()
        )
    except Exception as e:
        return f"Could not validate order: {e}. Ask the founder to clarify missing or invalid details."

    total_price = SERVICES[order.service]

    # Append to Google Sheet
    if ws:
        try:
            ws.append_row([
                order.client_name,
                order.startup_name,
                order.service,
                order.platform,
                order.target_date,
                total_price
            ])
            return f"Logged to Google Sheet: {order.service} for {order.client_name} ({order.startup_name}) targeting {order.platform} by {order.target_date}, Total: Rs {total_price}."
        except Exception as err:
            return f"Sheet error: {err}. Order validated locally for {order.client_name}, Total: Rs {total_price}."
    else:
        try:
            import csv
            csv_path = "campaign_orders.csv"
            file_exists = os.path.exists(csv_path)
            with open(csv_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                if not file_exists:
                    writer.writerow(["client_name", "startup_name", "service", "platform", "target_date", "total_price"])
                writer.writerow([order.client_name, order.startup_name, order.service, order.platform, order.target_date, total_price])
        except Exception:
            pass
        return f"Order confirmed and logged locally: {order.service} for {order.client_name} ({order.startup_name}) on {order.target_date}, Total: Rs {total_price}."

In [4]:
@tool
def search_knowledge(question: str) -> str:
    """Search agency policies, specifications, and FAQs: free revisions, turnaround times,
    9:16 vertical aspect ratio, audio rights, commercial monetization, NDA confidentiality,
    cancellation terms, voice cloning, thumbnails, retention rules, deliverables, and payment terms.
    Use for ANY question regarding agency rules, video specs, or policies, not for catalogue prices or hours."""
    results = faq_collection.query(query_texts=[question], n_results=3)
    return "\n".join(results["documents"][0])

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the AI Startup Marketing Agent for early-stage founders creating 9:16 vertical videos.
FACTS:  We produce high-retention faceless videos for YouTube Shorts and Instagram Reels.
RULES:  Always use get_service_price for pricing and check_operating_hours for timings; never guess them.
        For any policy, technical specification, or legal question, use search_knowledge and answer ONLY from what it returns.
        If the answer is not present in the knowledge base, say: "I'll check with the agency founders." Do not invent information.
        Before calling log_campaign_order, ensure you have: client name, startup name, service, platform, and target date.
FORMAT: Under 60 words, encouraging, practical, and clear."""

agent = create_agent(
    model=llm,
    tools=[get_service_price, check_operating_hours, log_campaign_order, search_knowledge],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver()
)

def chat(thread_id: str, text: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
        return result["messages"][-1].content
    except Exception as e:
        print("Agent error:", e)
        return "I encountered a brief issue. Please try your request again."

# Thread 1: Founder Sanjay asking about policies and video format
print("Sanjay (Thread 1):", chat("sanjay", "Hi, I'm Sanjay from DevPulse. Do you offer free revisions?"))
print("Sanjay (Thread 1):", chat("sanjay", "What video format and aspect ratio do you deliver?"))

# Thread 2: Founder Priya asking about cancellation (demonstrating session isolation)
print("\nPriya (Thread 2):", chat("priya", "Hello, I'm Priya from FinTrack. Can I cancel if I don't like the script?"))

# Thread 1: Sanjay returning — Testing Conversational Memory + RAG synthesis combined
print("\nSanjay (Thread 1 - Memory + RAG):", chat("sanjay", "What's my name and startup, and will I pay for revisions?"))

# Break it test: Asking an out-of-scope question to verify safe refusal
print("\nBreak It Test:", chat("sanjay", "Do you build custom mobile apps in Flutter?"))

Sanjay (Thread 1): Yes, every video order includes 2 rounds of free script and visual revisions. Additional revisions are Rs 150 each.
Sanjay (Thread 1): All videos are produced in 9:16 vertical format (1080x1920 resolution at 60 fps), optimized for YouTube Shorts, Instagram Reels, and TikTok.

Priya (Thread 2): Cancellations are 100% free before script drafting begins. Once video rendering starts, a 50% kill fee applies to cover GPU compute costs.

Sanjay (Thread 1 - Memory + RAG): You are Sanjay from DevPulse. Every video order includes 2 rounds of free script and visual revisions.

Break It Test: I'll check with the agency founders.


In [6]:
# Write Day 5 RAG and policy benchmark tests to tests.csv
with open("tests.csv", "w", encoding="utf-8") as f:
    f.write("""Do you offer free revisions?,2
What aspect ratio do you deliver?,9:16
Is the background music royalty-free?,royalty-free
How much is short script writing?,700
Can you sign an NDA?,NDA
Do you build mobile apps?,check with the agency founders
""")

print("Appended Day 5 test questions to tests.csv.")
print("Running Automated Evaluation Suite...")
print("-" * 85)
print(f"{'Question':<50} | {'Expected':<25} | {'Status'}")
print("-" * 85)

import csv

passed = 0
total = 0

with open("tests.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if not row or len(row) < 2:
            continue
        q, expected = row[0].strip(), row[1].strip()
        # Use a fresh thread per test to ensure clean independent evaluation
        resp = chat(f"eval_suite_{i}", q)
        status = "PASS" if expected.lower() in resp.lower() else "FAIL"
        if status == "PASS":
            passed += 1
        total += 1
        print(f"{q:<50} | {expected:<25} | {status}")

pass_rate = (passed / total * 100) if total else 0
print("-" * 85)
print(f"Results: {passed}/{total} Passed ({pass_rate:.1f}% Pass Rate)")
if pass_rate >= 80:
    print("Single-Agent Sign-Off Milestone: Achieved (Target: >= 80%)")
else:
    print("Review failed test cases to refine RULES in system prompt.")

Appended Day 5 test questions to tests.csv.
Running Automated Evaluation Suite...
-------------------------------------------------------------------------------------
Question                                           | Expected                  | Status
-------------------------------------------------------------------------------------
Do you offer free revisions?                       | 2                         | PASS
What aspect ratio do you deliver?                  | 9:16                      | PASS
Is the background music royalty-free?              | royalty-free              | PASS
How much is short script writing?                  | 700                       | PASS
Can you sign an NDA?                               | NDA                       | PASS
Do you build mobile apps?                          | check with the agency founders | PASS
-------------------------------------------------------------------------------------
Results: 6/6 Passed (100.0% Pass Rate)
Single-Agent